###  Activity 2.0: Loop Ordering and False Sharing

This activity reinforces the concept of reduction and the caching principles taught in the lecture on Cilk on Sep. 18. It is recommended that you run this on the CS machines `gradx.cs.jhu.edu` or `ugradx.cs.jhu.edu`.  The results make sense here.  It is OK to run this on any machine that has at least 4 cores.  If you run on a different machine, you may end up with slightly different results. It is OK if your results don't track exactly with the expected findings. On my M1 Apple Silicon laptop the results get confusing.

**Due date**: Tuesday October 1st, 2024, 5:00 pm EDT.

**Instructions for Submission**: Submit via Gradescope.

### The Program

This is a nested loop program that counts the number of occurences of a list of tokens in an array of elements. This is a common computing pattern in data analytics. This could be used to count the number of messages sent in a network from a set of sources.

There are two serial versions of the program.  These are:
  * `countTokensElementsFirst`: loop over the larger `elements` array in the outer loop and the smaller `tokens` array in the inner loop.
  * `countTokensTokensFirst`: loop over the smaller `tokens` array in the outer loop and the larger `elements` array in the inner loop.
  
This is not a 2-d dimensional data structure like our previous examples. It is 2 separate arrays.

#### Programming


Complete the *TODO* instructions in [ebook/activities/tokens_omp/activity2_tokens.cpp](https://github.com/randalburns/pcds.2024/blob/main/ebook/activities/tokens_omp/activity2_tokens.cpp)

1. Add `parallel for` directives to functions:
   * `omp_countTokensElementsFirst`
   * `omp_countTokensTokensFirst`
2. Add `parallel for` and `reduction` directive for the array `token_counts` for:
    * `omp_countTokensElementsFirst_reduce`
    * `omp_countTokensTokensFirst_reduce`
  
The array reduction clause was added to OpenMP and requires one to specify the length of the array.  A simple example is provided in https://dvalters.github.io/optimisation/code/2016/11/06/OpenMP-array_reduction.html.

3. Unroll the loop 8 times in `unroll_omp_countTokensElementsFirst_reduce`. You may assume the the `tokens` array is evenly divisible by 8.

On the `gradx.cs.jhu.edu` machine after I added this code, I got the timing results
```
Tokens First time: 8.07097 seconds
Elements First time: 6.93468 seconds
OMP Tokens First time: 2.10465 seconds
OMP Elements First time: 1.78919 seconds
OMP Tokens First Reduce time: 1.99353 seconds
OMP Elements First Reduce time.: 1.78073 seconds
Unroll OMP Elements First Reduce time.: 0.926184 seconds
```
building with the command line
<!-- ``` -->
> g++ -O0 -fopenmp activity2_tokens.cpp
<!-- ``` -->
Compiling with `-O0` turns off all compiler optimizations to prevent the compiler from making unknown optimizations that would confound our results.

#### Questions

Provide brief but complete answers to the following questions in the following cell.

1. Why is it more efficient to iterate over the `tokens` in the inner loop? 
(_Hint_: Access to both arrays is sequential. This is a question of memory access patterns, cache capacity and cache misses.)


2. Of the functions `omp_countTokensElementsFirst` and `omp_countTokensTokensFirst`:


    a. Which function performs <b><i>sharing</i></b> in the `tokens` array?


    b. Which function assigns different elements of the `token_counts` array to different threads?

    
3. For the function that assigns different tokens to different threads, how does <b><i>false sharing</b></i> arise? Be specific about the memory access pattern or include a drawing/schema.
4. For the unrolled loop, why is it more efficient? What computations and instructions are avoided?

#### Answers

<i>Include your answers to Questions 1-4 in this cell</i>

Question 1

1. The number tokens in this code is 128. This means that the token array is most likely small enough to fit in cache. On the other hand the number of elements is 4096*256. It is large and probably would not fit into a single cache. Thus when comparing items in the inner loop, if tokens array is in the inner loop, we wouldn't have cache miss for tokens and we would have a cache miss for elements. Call number of cache miss $n$. However, if the elements are in the inner loop, there will be $128n$ cache misses since the inner loop calls each element 128 times.
2. a: omp_countTokensElementsFirst. This is because "tok" is shared among each thread in the inner loop.
   b: omp_countTokensTokensFirst. The parallization of the outer loop sends each "tok", index of token_counts to different thread.
3. countTokensTokensFirst might have false sharing. Even though each tokens are assigned to different threads, token_counts can be saved within the same cache line. Thus when updating token_counts, the threads might be writing to the same cache line. In the event of this, the thread will need to send the cache back to the L3 cache(most likely, might be memory due to size) and the L3 cache will send the line to another thread.  ![False Sharing](false_sharing.png)
4. By unrolling, we can reduce the associated branching instruction. in a for loop, For each increment, we would need to compare counter to max length, increment, and loop back. However, by unrolling, we can reduce the number of comparisons and branching instructions as increment increases by 8.


<i>Copy-Paste your code from `activity2_tokens.cpp` to the cell below</i>

In [ ]:
// C++ Code goes here

#include <iostream>
#include <chrono>
#include <omp.h>

// initialize elements to random integer values 0 to range-1
void initElements (unsigned int range, unsigned int num_els, unsigned int* elements) {
    for (int i=0; i<num_els; i++)  {
        elements[i] = rand() % range;
    }
}

// initialize tokens to search.  again 0 to range-1
// note, we should probably enforce that tokens are unique. not important for performance.
void initTokens (int range, int num_toks, unsigned int* tokens) {
    for (int i=0; i<num_toks; i++)  {
        tokens[i] = rand() % range;
    }
}

// initialize all token counts to zero
void initCounts (int num_toks, unsigned int* token_counts) {
    for (int i=0; i<num_toks; i++)  {
        token_counts[i] = 0;
    }
}

// count the number of appearances of each token in the data
void countTokensElementsFirst (unsigned int num_els, unsigned int num_tokens,
                  unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {

    /* for all elements in the array */
    for (int el=0; el<num_els; el++) {
        /* for all tokens in the list */
        for (int tok=0; tok<num_tokens; tok++) {
            /* update the count for the token */
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }
        }
    }
}

// count the number of appearances of each token in the data
void countTokensTokensFirst (unsigned int num_els, unsigned int num_tokens,
                             unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {

    /* for all tokens in the list */
    for (int tok=0; tok<num_tokens; tok++) {
        /* for all elements in the array */
        for (int el=0; el<num_els; el++) {
            /* update the count for the token */
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }
        }
    }
}

// count the number of appearances of each token in the data
void omp_countTokensElementsFirst (unsigned int num_els, unsigned int num_tokens,
                  unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {

    //TODO parallel for
    /* for all elements in the array */
    #pragma omp parallel for
    for (int el=0; el<num_els; el++) {
        /* for all tokens in the list */
        for (int tok=0; tok<num_tokens; tok++) {
            /* update the count for the token */
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }
        }
    }
}

// count the number of appearances of each token in the data
void omp_countTokensTokensFirst (unsigned int num_els, unsigned int num_tokens,
                             unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {

    //TODO parallel for
    /* for all tokens in the list */
    #pragma omp parallel for 
    for (int tok=0; tok<num_tokens; tok++) {
        /* for all elements in the array */
        for (int el=0; el<num_els; el++) {
            /* update the count for the token */
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }
        }
    }
}

// elements first with reduction
void omp_countTokensElementsFirst_reduce (unsigned int num_els, unsigned int num_tokens,
                             unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {

    //TODO parallel for reduction
    /* for all elements in the array */
    #pragma omp parallel for reduction(+: token_counts[:num_tokens])
    for (int el=0; el<num_els; el++) {
        /* for all tokens in the list */
        for (int tok=0; tok<num_tokens; tok++) {
            /* update the count for the token */
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }
        }
    }
}

// tokens first with reduction
void omp_countTokensTokensFirst_reduce (unsigned int num_els, unsigned int num_tokens,
                             unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {

    //TODO parallel for reduction
    /* for all tokens in the list */
    #pragma omp parallel for reduction(+: token_counts[:num_tokens])
    for (int tok=0; tok<num_tokens; tok++) {
        /* for all elements in the array */
        for (int el=0; el<num_els; el++) {
            /* update the count for the token */
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }
        }
    }
}


// unroll tokens elements first with reduction


void unroll_omp_countTokensElementsFirst_reduce (unsigned int num_els, unsigned int num_tokens,
                             unsigned int* elements, unsigned int* tokens, unsigned int* token_counts) {
    //TODO parallel for reduction 
    /* for all elements in the array */
    #pragma omp parallel for reduction(+: token_counts[:num_tokens])
    for (int el=0; el<num_els; el++) {
        for (int tok=0; tok < num_tokens; tok= tok + 8) {
            if (elements[el] == tokens[tok]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 1]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 2]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 3]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 4]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 5]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 6]) {
                token_counts[tok]++;
            }

            if (elements[el] == tokens[tok + 7]) {
                token_counts[tok]++;
            }
        }
    }
}

int main() {

    const unsigned int range = 4096;
    const unsigned int num_tokens = 128;
    const unsigned int num_elements = 4096*256;
    const unsigned int loop_iterations = 16;

    unsigned int tokens[num_tokens];
    unsigned int elements[num_elements];
    unsigned int token_counts[num_tokens];

    initElements(range, num_elements, elements);
    initTokens(range, num_tokens, tokens);
    initCounts(num_tokens, token_counts);

    omp_set_num_threads(4);

    // run once to warm the cache
    countTokensTokensFirst(num_elements, num_tokens, elements, tokens, token_counts);

    // countTokensTokensFirst
    // Start the timer
    auto start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        countTokensTokensFirst(num_elements, num_tokens, elements, tokens, token_counts);
    }
    // Stop the timer
    auto end = std::chrono::high_resolution_clock::now();
    // Calculate the duration
    std::chrono::duration<double> duration = end - start;
    // Print the duration in seconds
    std::cout << "Tokens First time: " << duration.count() << " seconds" << std::endl;

    // reset counts only works right if running one loop_iteration
    initCounts(num_tokens, token_counts);

    // run once to warm the cache
    countTokensElementsFirst(num_elements, num_tokens, elements, tokens, token_counts);
  

    // countTokensElementsFirst
    start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        countTokensElementsFirst(num_elements, num_tokens, elements, tokens, token_counts);
    }
    end = std::chrono::high_resolution_clock::now();
    duration = end - start;
    std::cout << "Elements First time: " << duration.count() << " seconds" << std::endl;

    // reset counts only works right if running one loop_iteration
    initCounts(num_tokens, token_counts);
    	
    
    // omp_countTokensTokensFirst
    start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        omp_countTokensTokensFirst(num_elements, num_tokens, elements, tokens, token_counts);
    }
    end = std::chrono::high_resolution_clock::now();
    duration = end - start;
    std::cout << "OMP Tokens First time: " << duration.count() << " seconds" << std::endl;

    // reset counts only works right if running one loop_iteration
    initCounts(num_tokens, token_counts);

    
    // omp_countTokensElementsFirst
    start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        omp_countTokensElementsFirst(num_elements, num_tokens, elements, tokens, token_counts);
    }
    end = std::chrono::high_resolution_clock::now();
    duration = end - start;
    std::cout << "OMP Elements First time: " << duration.count() << " seconds" << std::endl;

    // reset counts only works right if running one loop_iteration
    initCounts(num_tokens, token_counts);

    // omp_countTokensTokensFirst_reduce
    start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        omp_countTokensTokensFirst_reduce(num_elements, num_tokens, elements, tokens, token_counts);
    }
    end = std::chrono::high_resolution_clock::now();
    duration = end - start;
    std::cout << "OMP Tokens First Reduce time: " << duration.count() << " seconds" << std::endl;

    // omp_countTokensElementsFirst_reduce
    start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        omp_countTokensElementsFirst_reduce(num_elements, num_tokens, elements, tokens, token_counts);
    }
    end = std::chrono::high_resolution_clock::now();
    duration = end - start;
    std::cout << "OMP Elements First Reduce time.: " << duration.count() << " seconds" << std::endl;

    // reset counts only works right if running one loop_iteration
    initCounts(num_tokens, token_counts);

    // unroll_omp_countTokensElementsFirst_reduce
    start = std::chrono::high_resolution_clock::now();
    for(int j=0; j<loop_iterations; j++) {
        unroll_omp_countTokensElementsFirst_reduce(num_elements, num_tokens, elements, tokens, token_counts);
    }
    end = std::chrono::high_resolution_clock::now();
    duration = end - start;
    std::cout << "Unroll OMP Elements First Reduce time.: " << duration.count() << " seconds" << std::endl;
}
